# Adilet Search — final inference demonstration

Run the Docker prototype first (`infra/start-final.ps1 -Index`). This notebook calls the real API; it does not use mocked responses or load credentials. The deployed E5 and cross-encoder models are pretrained. This is an integration demonstration, not evidence of our own fine-tuning or a human-gold quality benchmark.

The current demo index covers the official Labor Code in RU/KK. See the readiness report for full-corpus, model-selection, load and public-hosting limitations.

In [1]:
import json
import urllib.request
from urllib.error import HTTPError

BASE_URL = "http://127.0.0.1:18000"

def api(path, payload=None):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(BASE_URL + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.load(response)

health = api("/api/v1/health")
version = api("/api/v1/version")
assert health["status"] == "ok", health
assert version["pipeline_version"] == "0.2.0", version
print(json.dumps({"health": health, "version": version}, indent=2, ensure_ascii=False))

{
  "health": {
    "status": "ok",
    "components": {
      "database": "ok",
      "qdrant": "ok",
      "ml_service": "ok",
      "llm": "ok"
    },
    "pipeline_version": "0.2.0",
    "app_version": "0.1.0"
  },
  "version": {
    "app_version": "0.1.0",
    "git_sha": "dev",
    "pipeline_version": "0.2.0",
    "index_collection": "legal_chunks__0.2.0"
  }
}


## New Russian and Kazakh questions

Check the retrieved articles and degradation flags. A correct result on these source-derived examples does not establish general accuracy.

In [2]:
examples = [
    ("ru", "Какова нормальная продолжительность рабочего времени?", "K1500000414:ru:a68"),
    ("kk", "Жыл сайынғы ақы төленетін негізгі еңбек демалысының ұзақтығы қандай?", "K1500000414:kk:a88"),
]
for lang, question, expected in examples:
    result = api("/api/v1/search", {"query": question, "lang": lang, "mode": "hybrid", "top_k": 3})
    ids = [item["article"]["article_id"] for item in result["results"]]
    print(json.dumps({"language": lang, "query": question, "article_ids": ids, "expected_rank": ids.index(expected) + 1 if expected in ids else None, "degraded": result["degraded"]}, ensure_ascii=False, indent=2))
    assert ids and ids[0] == expected
    assert not result["degraded"]

{
  "language": "ru",
  "query": "Какова нормальная продолжительность рабочего времени?",
  "article_ids": [
    "K1500000414:ru:a68",
    "K1500000414:ru:a67",
    "K1500000414:ru:a69"
  ],
  "expected_rank": 1,
  "degraded": []
}


{
  "language": "kk",
  "query": "Жыл сайынғы ақы төленетін негізгі еңбек демалысының ұзақтығы қандай?",
  "article_ids": [
    "K1500000414:kk:a88",
    "K1500000414:kk:a92",
    "K1500000414:kk:a90"
  ],
  "expected_rank": 1,
  "degraded": []
}


## Inspect the stored source

Legal text is retrieved verbatim. The dated snapshot does not establish that every later amendment is included.

In [3]:
article = api("/api/v1/articles/K1500000414:ru:a68")
print(article["text"])
print("Official source:", article["article"]["url"])
assert "40" in article["text"]

Нормальная продолжительность рабочего времени не должна превышать 40 часов в неделю.
В трудовом договоре может предусматриваться меньшая продолжительность рабочего времени с оплатой как за нормальную продолжительность рабочего времени.
Суммарная продолжительность ежедневной работы по месту основной работы и работы по совместительству не должна превышать норму продолжительности ежедневной работы, установленную  пунктом 4  статьи 71 настоящего Кодекса, более чем на 4 часа.
Работникам, осуществляющим уход за больным членом семьи в соответствии с медицинским заключением, одному из родителей (усыновителю, удочерителю), имеющих ребенка (детей) в возрасте до трех лет, одинокому родителю, воспитывающему ребенка в возрасте до четырнадцати лет (ребенка с инвалидностью до восемнадцати лет), иным лицам, воспитывающим указанную категорию детей без родителя, учащимся для выполнения в свободное от учебы время работы, не причиняющей вреда здоровью и не нарушающей процесса обучения, работникам, которым

## Stream an answer and validate reference IDs

Generation mode is a server setting: extractive fallback or an external OpenAI-compatible LLM. Check `final_runtime.json` for the actual mode. A valid citation reference is not proof that the source supports every claim.

In [4]:
payload = {"query": examples[0][1], "lang": "ru"}
request = urllib.request.Request(BASE_URL + "/api/v1/answer", data=json.dumps(payload, ensure_ascii=False).encode("utf-8"), headers={"Content-Type": "application/json", "Accept": "text/event-stream"})
events = []
event = None
with urllib.request.urlopen(request, timeout=120) as response:
    for raw_line in response:
        line = raw_line.decode("utf-8").strip()
        if line.startswith("event: "):
            event = line[7:]
        elif line.startswith("data: "):
            events.append((event, json.loads(line[6:])))
sources = next(data["sources"] for event, data in events if event == "sources")
done = next(data for event, data in events if event == "done")
print(done["text"])
print("Citation references:", done["citations"])
print("Pipeline:", done["pipeline_version"])
assert set(done["citations"]) <= {source["ref"] for source in sources}
assert done["text"]

Нормальная продолжительность рабочего времени не должна превышать 40 часов в неделю. При этом в трудовом договоре может быть предусмотрена меньшая продолжительность рабочего времени с оплатой как за нормальную продолжительность рабочего времени [1].
Citation references: [1]
Pipeline: 0.2.0


## Invalid input is rejected

The API should reject an empty question rather than attempting inference.

In [5]:
try:
    api("/api/v1/search", {"query": ""})
except HTTPError as error:
    body = json.load(error)
    print(error.code, body["error"]["code"])
    assert error.code == 422
else:
    raise AssertionError("Empty question was accepted")

422 validation_error
